In [27]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
import re
import numpy as np
import sys 

# Path to Environment resources
env_path = f"{notebookutils.nbResPath}/env"

if env_path not in sys.path:
    sys.path.append(env_path)

from udf_sharepoint import download_file_from_sharepoint, upload_file_2_sharepoint # download_file_from_sharepoint(sp_folder_name, file_name, lakehouse_folder_path) ,  upload_file_2_sharepoint(full_path_2_filename,sp_folder_name)
from data_cleaning_utils import auto_cast_string_columns, clean_columns


StatementMeta(, f59fd11e-e3fe-4112-82b3-c15a00f5df19, 31, Finished, Available, Finished, False)

In [28]:
df_scorecard = spark.table("srm.prophet_scorecard").toPandas()

StatementMeta(, f59fd11e-e3fe-4112-82b3-c15a00f5df19, 32, Finished, Available, Finished, False)

In [29]:
# display(df_scorecard)

StatementMeta(, f59fd11e-e3fe-4112-82b3-c15a00f5df19, 33, Finished, Available, Finished, False)

In [30]:
# ── Load the table ───────────────────────────────────────────────
df_scorecard = spark.table("srm.prophet_scorecard").toPandas()


# ── Enforce a fixed, deterministic row order — Spark doesn't guarantee ─
# the order rows come back in after a save/reload round-trip, so this
# has to be re-applied here regardless of what order Notebook 5 built.
INDICATOR_DISPLAY_ORDER = [
    "Global Stock-to-Use Ratio",
    "Import Demand Pressure",
    "Production Potential Index",
    "KSA Import Concentration",
    "Export Restriction Status",
    "Logistic Disruption Index",
    "Domestic Strategic Reserves",
    "FINAL RISK SCORE",   # composite row — always last
]

COMMODITY_ORDER = ["Wheat", "Corn", "Rice", "Soybean", "Barley"]

df_scorecard["Variable"] = pd.Categorical(
    df_scorecard["Variable"], categories=INDICATOR_DISPLAY_ORDER, ordered=True
)
df_scorecard["Commodity"] = pd.Categorical(
    df_scorecard["Commodity"], categories=COMMODITY_ORDER, ordered=True
)

df_scorecard = df_scorecard.sort_values(["Commodity", "Variable"]).reset_index(drop=True)

# ── Convert back to plain strings — Categorical dtype breaks the later ─
# .fillna("") call below, since "" was never declared as one of the
# categories. Row order is already locked in by this point.
df_scorecard["Variable"] = df_scorecard["Variable"].astype(str)
df_scorecard["Commodity"] = df_scorecard["Commodity"].astype(str)

# ── Update this one line each month ────────────────────────────
CURRENT_MONTH = pd.Timestamp("2026-07-01")

# ── Add Current_Month column in YYYYMM format ──────────────────
df_scorecard["Current_Month"] = CURRENT_MONTH.strftime("%Y%m")

# ── Normalize any stray "nan"/"None"/"NaT" TEXT to real NaN first ─
def normalize_missing(val):
    if isinstance(val, str) and val.strip().lower() in ("nan", "none", "nat", ""):
        return np.nan
    return val

# ── Strip "(Watch)"/"(Low)"/etc. text from period columns, keep just the number ─
def extract_numeric(val):
    if pd.isna(val):
        return val
    if isinstance(val, str):
        match = re.match(r'^\s*(-?\d+\.?\d*)', val)
        if match:
            return float(match.group(1))
    return val

for col in ["Current", "M1", "M2", "M3"]:
    df_scorecard[col] = df_scorecard[col].apply(extract_numeric)

df_scorecard = df_scorecard.applymap(normalize_missing)

# ── Replace NaN with blank for a clean Excel export ─────────────
df_scorecard = df_scorecard.fillna("")

# ── Determine chokepoint status — "Chokepoint" if ANY strait is fully closed ─
df_severity_check = spark.table("srm.prophet_chokepoint_severity").toPandas()
any_full_closure = (df_severity_check["severity"] >= 1.0).any()
chokepoint_status = "Chokepoint" if any_full_closure else "Normal"

df_scorecard["Chokepoint_Indicator"] = chokepoint_status


# ── Output path — Lakehouse Files section, downloadable from there ─
output_path = "/lakehouse/default/Files/prophet_scorecard_export.xlsx"

commodities = ["Wheat", "Corn", "Rice", "Soybean", "Barley"]

StatementMeta(, f59fd11e-e3fe-4112-82b3-c15a00f5df19, 34, Finished, Available, Finished, False)

/tmp/ipykernel_17079/2397760882.py:61: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_scorecard = df_scorecard.applymap(normalize_missing)


In [31]:
# with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
#     # Sheet 1 — all records
#     df_scorecard.to_excel(writer, sheet_name="All Commodities", index=False)

#     # Sheets 2-6 — one per commodity
#     for commodity in commodities:
#         df_c = df_scorecard[df_scorecard["Commodity"] == commodity]
#         df_c.to_excel(writer, sheet_name=commodity, index=False)

# print(f"✓ Excel file saved: {output_path}")
# print(f"  Current_Month: {CURRENT_MONTH.strftime('%Y%m')}")
# print(f"  Chokepoint_Indicator: {chokepoint_status}")
# print(f"  Columns: {df_scorecard.columns.tolist()}")
# print(f"  Sheet 'All Commodities': {len(df_scorecard)} rows")
# for commodity in commodities:
#     n = len(df_scorecard[df_scorecard["Commodity"] == commodity])
#     print(f"  Sheet '{commodity}': {n} rows")

# upload_file_2_sharepoint(output_path, "EWS_Datasets/Final_Model_Results")
# print("Final Model Excel uploaded to SharePoint from", output_path)

StatementMeta(, f59fd11e-e3fe-4112-82b3-c15a00f5df19, 35, Finished, Available, Finished, False)

In [32]:
import os

# ── Append this run's data to the existing file instead of overwriting ─
file_exists = os.path.exists(output_path)

if file_exists:
    print(f"Existing file found — appending {chokepoint_status} scenario data")

    # Read existing sheets into memory
    existing_all = pd.read_excel(output_path, sheet_name="All Commodities")
    existing_by_commodity = {
        c: pd.read_excel(output_path, sheet_name=c) for c in commodities
    }

    # ── Dedup key — if this exact scenario/month/commodity/indicator combo
    # already exists (e.g. rerunning the same scenario twice), replace the
    # old rows rather than duplicating them. Otherwise, append fresh.
    dedup_cols = ["Commodity", "Domain", "Variable", "Current_Month", "Chokepoint_Indicator"]

    def merge_replace(old_df, new_df, keys):
        if old_df.empty:
            return new_df

        # ── Normalize dtypes on the merge keys — Excel round-trip can silently
        # convert "202607" (string) into 202607 (int) on read-back, breaking
        # the merge even though the values are logically identical.
        old_df = old_df.copy()
        new_df = new_df.copy()
        for k in keys:
            old_df[k] = old_df[k].astype(str)
            new_df[k] = new_df[k].astype(str)

        merge_check = old_df.merge(new_df[keys].drop_duplicates(), on=keys, how="left", indicator=True)
        old_kept = old_df[merge_check["_merge"].values == "left_only"]
        return pd.concat([old_kept, new_df], ignore_index=True)

    combined_all = merge_replace(existing_all, df_scorecard, dedup_cols)

else:
    print("No existing file — creating fresh")
    combined_all = df_scorecard.copy()

# ── Write the combined data back out ────────────────────────────
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    combined_all.to_excel(writer, sheet_name="All Commodities", index=False)

    for commodity in commodities:
        df_c = combined_all[combined_all["Commodity"] == commodity]
        df_c.to_excel(writer, sheet_name=commodity, index=False)

print(f"✓ Excel file saved: {output_path}")
print(f"  Current_Month: {CURRENT_MONTH.strftime('%Y%m')}")
print(f"  Chokepoint_Indicator: {chokepoint_status}")
print(f"  Columns: {combined_all.columns.tolist()}")
print(f"  Sheet 'All Commodities': {len(combined_all)} rows (across all scenarios run so far)")
for commodity in commodities:
    n = len(combined_all[combined_all["Commodity"] == commodity])
    print(f"  Sheet '{commodity}': {n} rows")

upload_file_2_sharepoint(output_path, "EWS_Datasets/Final_Model_Results")
print("Final Model Excel uploaded to SharePoint from", output_path)

StatementMeta(, f59fd11e-e3fe-4112-82b3-c15a00f5df19, 36, Finished, Available, Finished, False)

Existing file found — appending Chokepoint scenario data
✓ Excel file saved: /lakehouse/default/Files/prophet_scorecard_export.xlsx
  Current_Month: 202607
  Chokepoint_Indicator: Chokepoint
  Columns: ['Commodity', 'Domain', 'Variable', 'Direction', 'Unit', 'Weight_pct', 'Baseline', 'Current', 'M1', 'M2', 'M3', 'Score', 'Weighted_Avg', 'Low_Good', 'Watch', 'Warning', 'Emergency', 'Writeup_Current', 'Writeup_M1', 'Writeup_M2', 'Writeup_M3', 'Current_Month', 'Chokepoint_Indicator']
  Sheet 'All Commodities': 80 rows (across all scenarios run so far)
  Sheet 'Wheat': 16 rows
  Sheet 'Corn': 16 rows
  Sheet 'Rice': 16 rows
  Sheet 'Soybean': 16 rows
  Sheet 'Barley': 16 rows
{'@odata.context': "https://graph.microsoft.com/v1.0/$metadata#sites('41577faa-cc80-40ec-9cd9-e78e9d25b512%2C410ab16c-260a-42aa-b175-e60887a1705c')/drives('b%21qn9XQYDM7ECc2eeOnSW1EmyxCkEKJqpCsXXmCIehcFyZKrUC4Xq6Qrfw8isB_dqp')/items/$entity", '@microsoft.graph.downloadUrl': 'https://salic.sharepoint.com/sites/Research

In [33]:
# with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
#     # Sheet 1 — all records
#     df_scorecard.to_excel(writer, sheet_name="All Commodities", index=False)

#     # Sheets 2-6 — one per commodity
#     for commodity in commodities:
#         df_c = df_scorecard[df_scorecard["Commodity"] == commodity]
#         df_c.to_excel(writer, sheet_name=commodity, index=False)

# print(f"✓ Excel file saved: {output_path}")
# print(f"  Current_Month: {CURRENT_MONTH.strftime('%Y%m')}")
# print(f"  Sheet 'All Commodities': {len(df_scorecard)} rows")
# for commodity in commodities:
#     n = len(df_scorecard[df_scorecard["Commodity"] == commodity])
#     print(f"  Sheet '{commodity}': {n} rows")

StatementMeta(, f59fd11e-e3fe-4112-82b3-c15a00f5df19, 37, Finished, Available, Finished, False)